In [2]:
import pandas as pd
import numpy as np
import warnings

In [3]:
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv('spotify_mult_genre.csv')
df

,artists,track_name,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity,title_size,duration,num_artists
0,!nvite,pagadoff,0,0.784,0.657,11,-7.591,0,0.3480,0.332,0.00362,0.1310,0.501,84.997,4,['study'],5,8,2.264333,1
1,!nvite,strolling,0,0.857,0.381,2,-12.755,1,0.1920,0.666,0.01910,0.1260,0.329,84.997,4,['study'],41,9,2.314583,1
2,"""Puppy Dog Pals"" Cast",Going on a Mission,0,0.629,0.776,7,-3.839,0,0.0470,0.021,0.00000,0.0930,0.957,93.937,3,['children'],55,18,0.635733,1
3,"""Puppy Dog Pals"" Cast",Puppy Dog Pals Main Title Theme,0,0.781,0.936,3,-4.709,1,0.2020,0.171,0.00141,0.2020,0.873,182.148,4,['children'],60,31,0.963150,1
4,"""Weird Al"" Yankovic","Amish Paradise (Parody of ""Gangsta's Paradise""...",0,0.728,0.448,8,-10.540,1,0.1720,0.103,0.00000,0.2670,0.483,80.902,4,['comedy'],58,57,3.382000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83800,é»å¦,æº«æçæå¨,0,0.549,0.450,10,-7.176,0,0.0251,0.461,0.00000,0.1030,0.234,90.062,3,['mandopop'],23,15,4.856667,1
83801,é»å°ç¥,æ²é£éº½ç°¡å®,0,0.334,0.431,6,-5.861,1,0.0347,0.672,0.00000,0.1010,0.212,201.701,3,['mandopop'],56,15,5.168883,1
83802,é»æè¯,å ¤å²¸,0,0.549,0.478,0,-6.186,1,0.0276,0.783,0.00000,0.2550,0.341,125.917,4,['cantopop'],20,6,3.645550,1
83803,é¾èRyuzo,ã²ã²ã²ã®é¬¼å¤ªé (Instrumental),0,0.571,0.325,7,-11.442,1,0.0443,0.404,0.41700,0.0872,0.320,116.457,3,['guitar'],23,36,2.802283,1


In [5]:
# преобразуем столбец track_genre из строки в список
import ast

df['track_genre'] = df['track_genre'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

В части исследования, где был произведен EDA, показано, что большинство числовых переменных имеют максимальные значения равные 1, размах целевой переменной (popularity) равен 100 + были удалены выбросы, поэтому можно не стандартизировать данные.

Число наблюдений велико по сравнению с числом столбцов (признаков), поэтому можно применить one-hot-encoding для преобразования категориальной переменной track_genre. Для проверки гипотез была использована выборка, в которой треки принадлежали лишь только одному жанру, однако в данной части работы (моделирование), наоборот, для максимального сохранения информации о каждом объекте будут использованы все треки, вне зависимости от числа жанров.

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

In [7]:
mlb = MultiLabelBinarizer()
# создаём one-hot признаки жанров

genres_encoded = mlb.fit_transform(df['track_genre'])

genres_df = pd.DataFrame(genres_encoded, columns = mlb.classes_, index = df.index)

df = pd.concat([df, genres_df], axis=1)
df = df.drop(columns = 'track_genre')
df

,artists,track_name,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,spanish,study,swedish,synth-pop,tango,techno,trance,trip-hop,turkish,world-music
0,!nvite,pagadoff,0,0.784,0.657,11,-7.591,0,0.3480,0.332,...,0,1,0,0,0,0,0,0,0,0
1,!nvite,strolling,0,0.857,0.381,2,-12.755,1,0.1920,0.666,...,0,1,0,0,0,0,0,0,0,0
2,"""Puppy Dog Pals"" Cast",Going on a Mission,0,0.629,0.776,7,-3.839,0,0.0470,0.021,...,0,0,0,0,0,0,0,0,0,0
3,"""Puppy Dog Pals"" Cast",Puppy Dog Pals Main Title Theme,0,0.781,0.936,3,-4.709,1,0.2020,0.171,...,0,0,0,0,0,0,0,0,0,0
4,"""Weird Al"" Yankovic","Amish Paradise (Parody of ""Gangsta's Paradise""...",0,0.728,0.448,8,-10.540,1,0.1720,0.103,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83800,é»å¦,æº«æçæå¨,0,0.549,0.450,10,-7.176,0,0.0251,0.461,...,0,0,0,0,0,0,0,0,0,0
83801,é»å°ç¥,æ²é£éº½ç°¡å®,0,0.334,0.431,6,-5.861,1,0.0347,0.672,...,0,0,0,0,0,0,0,0,0,0
83802,é»æè¯,å ¤å²¸,0,0.549,0.478,0,-6.186,1,0.0276,0.783,...,0,0,0,0,0,0,0,0,0,0
83803,é¾èRyuzo,ã²ã²ã²ã®é¬¼å¤ªé (Instrumental),0,0.571,0.325,7,-11.442,1,0.0443,0.404,...,0,0,0,0,0,0,0,0,0,0


В качестве метрики качества моделей будем использовать RMSE.

Разбиваем выборку на две части: обучающую и тестовую, дабы иметь возможность оценить и сравнить получившиеся модели на новых данных:

In [8]:
from sklearn.model_selection import train_test_split

# разбиение производится в соотношении 80%:20%

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['artists', 'track_name', 'popularity']), df['popularity'], test_size=0.2, random_state = 29)

### Линейная регрессия

In [34]:
from sklearn.linear_model import LinearRegression

model_regression = LinearRegression(fit_intercept = False)
model_regression.fit(X_train, y_train)

y_pred_train = model_regression.predict(X_train)
y_pred_test = model_regression.predict(X_test)

In [9]:
from sklearn.metrics import mean_squared_error

In [36]:
RMSE_test = np.sqrt(mean_squared_error(y_test, y_pred_test)) 
RMSE_train = np.sqrt(mean_squared_error(y_train, y_pred_train)) 

print(f" RMSE test = {RMSE_test:.3f}; RMSE train = {RMSE_train:.3f}")

 RMSE test = 15.376; RMSE train = 15.591


Получается, что на новых данных в среднем построенная модель линейной регрессии ошибается в предсказании индекса популярности треков на 15.376 единиц, а качество модели на тестовой выборке меньше качества на трейне (нет явных оснований предполагать переобучение). Зметим, что ошибка прогноза, на самом деле, значительна, так как целевая переменная принадлежит промежутку [0; 100].

### Бэггинг

Сначала подберем оптимальные гиперпараметры:

In [10]:
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import GridSearchCV

In [44]:
np.random.seed(140)

n_estimators = [1, 10, 20, 50, 80, 100]
max_features = [1, 5, 10, 15, 20]

param_grid = {
    'n_estimators': n_estimators,
    'max_features': max_features
}

grid_search = GridSearchCV(
    BaggingRegressor(),
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error'
)
grid_search.fit(X_train, y_train)

print("Best estimator:", grid_search.best_estimator_)
print("Best RMSE (CV):", -grid_search.best_score_)

Best estimator: BaggingRegressor(max_features=20, n_estimators=80)
Best RMSE (CV): 18.011236834300032


In [45]:
np.random.seed(140)

bag = BaggingRegressor(n_estimators = 80, max_features = 20)
bag.fit(X_train, y_train)

y_pred_train = bag.predict(X_train)
y_pred_test = bag.predict(X_test)

RMSE_test = np.sqrt(mean_squared_error(y_test, y_pred_test)) 
RMSE_train = np.sqrt(mean_squared_error(y_train, y_pred_train)) 

print(f" RMSE test = {RMSE_test:.3f}; RMSE train = {RMSE_train:.3f}")

 RMSE test = 17.758; RMSE train = 11.776


В сравнении с линейной регрессией (базовой моделью), удалось снизить ошибку на обучающей выборке, однако на тестовой - RMSE выросла до 17.758, соотвественно, бэггинг с оптимальными параметрами проигрывает регрессии по качеству.

### Random Forest

Сначала аналогичным образом подберем оптимальные гиперпараметры:

In [11]:
from sklearn.ensemble import RandomForestRegressor

In [12]:
np.random.seed(140)

max_depth_array = [1, 10, 20, 50, 80]
min_samples_split_array = [2, 5, 10, 15, 20]
choices = []

param_grid = {
    'max_depth': max_depth_array,
    'min_samples_split': min_samples_split_array
}

grid_search = GridSearchCV(RandomForestRegressor(n_estimators = 50),
                           param_grid=param_grid, cv = 3, scoring = 'neg_root_mean_squared_error', n_jobs = -1)
grid_search.fit(X_train, y_train)
print(grid_search.best_estimator_)

RandomForestRegressor(max_depth=80, min_samples_split=20, n_estimators=50)


In [13]:
model_rfr = RandomForestRegressor(n_estimators = 80, max_depth = 80, min_samples_split = 20)

model_rfr.fit(X_train, y_train)

y_pred_train = model_rfr.predict(X_train)
y_pred_test = model_rfr.predict(X_test)

rfr_test = np.sqrt(mean_squared_error(y_test, y_pred_test)) 
rfr_train = np.sqrt(mean_squared_error(y_train, y_pred_train)) 

print(f" RMSE test = {rfr_test:.3f}; RMSE train = {rfr_train:.3f}")

 RMSE test = 14.344; RMSE train = 10.049


Заметим, что ошибка на тестовой выборке снизилась в сравнении с моделями линейной регрессии и бэггинга, поэтому случайный лес с оптимальными гиперпараметрами можно объявить лучшей моделью с точки зрения прогнозирования популярности треков. 

Более высокое качество случаного леса может быть объяснено следующими факторами:
1) Нелинейность влияния признаков на целевую переменную: популярность трека формируется под влиянием сложных музыкальных и контентных характеристик, а эти зависимости нелинейны и включают взаимодействия между признаками, например, влияние loudness зависит от жанра;

2) Бэггинг использует случайные подвыборки данных, но не вносит дополнительной случайности в выбор признаков. Деревья получаются сильно коррелированными → хуже обобщение. Random Forest добавляет случайный выбор признаков при каждом разбиении → больше разнообразия → меньше дисперсия → выше качество.

3) Большое количество категориальных признаков (жанры): после кодирования жанров появляются десятки–сотни бинарных переменных, а их влияние часто нелинейно и неоднородно. Random Forest эффективнее работает с таким типом данных.